# DATA SPLITTING AND MODEL TRAINING

In [ ]:
# Temporal Train / Test Split

# Train: 2008-2019, Validation: 2020, Test: 2021-2022
df_processed = spark.read.parquet(PROCESSED_PATH)

train_df = df_processed.filter(col("year") <= 2019).select("features", "arrest")
val_df   = df_processed.filter(col("year") == 2020).select("features", "arrest")
test_df  = df_processed.filter(col("year") >= 2021).select("features", "arrest")

train_df.cache()
val_df.cache()
test_df.cache()

print("Data Split Summary (Temporal)")
print("-" * 40)
print(f"Train (2008-2019) : {train_df.count():,} rows")
print(f"Validation (2020) : {val_df.count():,} rows")
print(f"Test  (2021-2022) : {test_df.count():,} rows")

# Check class balance in train
pos = train_df.filter(col("arrest") == 1).count()
neg = train_df.filter(col("arrest") == 0).count()
print(f"\nTrain class balance:")
print(f"  Arrest=1 : {pos:,} ({pos/(pos+neg)*100:.1f}%)")
print(f"  Arrest=0 : {neg:,} ({neg/(pos+neg)*100:.1f}%)")

Data Split Summary (Temporal)
----------------------------------------
Train (2008-2019) : 3,797,546 rows
Validation (2020) : 212,700 rows
Test  (2021-2022) : 449,652 rows

Train class balance:
  Arrest=1 : 960,768 (25.3%)
  Arrest=0 : 2,836,778 (74.7%)


In [ ]:
# Helper to save model safely

import shutil

def save_model(model, path):
    if os.path.exists(path):
        shutil.rmtree(path)
    model.save(path)
    print(f"Model saved to {path}")

In [ ]:
# Evaluation helper function

def evaluate_model(predictions, model_name):
    binary_eval = BinaryClassificationEvaluator(
        labelCol="arrest",
        rawPredictionCol="rawPrediction"
    )
    multi_eval = MulticlassClassificationEvaluator(
        labelCol="arrest",
        predictionCol="prediction"
    )

    auc       = binary_eval.evaluate(predictions)
    accuracy  = multi_eval.evaluate(predictions, {multi_eval.metricName: "accuracy"})
    f1        = multi_eval.evaluate(predictions, {multi_eval.metricName: "f1"})
    precision = multi_eval.evaluate(predictions, {multi_eval.metricName: "weightedPrecision"})
    recall    = multi_eval.evaluate(predictions, {multi_eval.metricName: "weightedRecall"})

    print(f"\nModel     : {model_name}")
    print(f"AUC-ROC   : {auc:.4f}")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")

    return {
        "model"     : model_name,
        "auc"       : round(auc, 4),
        "accuracy"  : round(accuracy, 4),
        "f1"        : round(f1, 4),
        "precision" : round(precision, 4),
        "recall"    : round(recall, 4)
    }


# MODEL 1: Logistic Regression

In [ ]:
lr = LogisticRegression(
    labelCol="arrest",
    featuresCol="features",
    maxIter=20,
    regParam=0.01,
    elasticNetParam=0.0
)

start    = time.time()
lr_model = lr.fit(train_df)
lr_preds = lr_model.transform(test_df)
lr_time  = round(time.time() - start, 2)

lr_results = evaluate_model(lr_preds, "Logistic Regression")
lr_results["train_time_sec"] = lr_time
print(f"Training time : {lr_time}s")
save_model(lr_model, f"{MODELS_DIR}/lr_model")


Model     : Logistic Regression
AUC-ROC   : 0.8806
Accuracy  : 0.9181
F1 Score  : 0.9126
Precision : 0.9112
Recall    : 0.9181
Training time : 41.8s
Model saved to /content/chicago_crimes/models/lr_model


# MODEL 2: Random Forest

In [ ]:
rf = RandomForestClassifier(
    labelCol="arrest",
    featuresCol="features",
    numTrees=50,
    maxDepth=10,
    seed=42
)

start    = time.time()
rf_model = rf.fit(train_df)
rf_preds = rf_model.transform(test_df)
rf_time  = round(time.time() - start, 2)

rf_results = evaluate_model(rf_preds, "Random Forest")
rf_results["train_time_sec"] = rf_time
print(f"Training time : {rf_time}s")
save_model(rf_model, f"{MODELS_DIR}/rf_model")


Model     : Random Forest
AUC-ROC   : 0.8400
Accuracy  : 0.9232
F1 Score  : 0.9087
Precision : 0.9255
Recall    : 0.9232
Training time : 951.79s
Model saved to /content/chicago_crimes/models/rf_model


# MODEL 3: Gradient Boosted Trees

In [ ]:
gbt = GBTClassifier(
    labelCol="arrest",
    featuresCol="features",
    maxIter=20,
    maxDepth=5,
    seed=42
)[]

start     = time.time()
gbt_model = gbt.fit(train_df)
gbt_preds = gbt_model.transform(test_df)
gbt_time  = round(time.time() - start, 2)

gbt_results = evaluate_model(gbt_preds, "Gradient Boosted Trees")
gbt_results["train_time_sec"] = gbt_time
print(f"Training time : {gbt_time}s")
save_model(gbt_model, f"{MODELS_DIR}/gbt_model")


Model     : Gradient Boosted Trees
AUC-ROC   : 0.8616
Accuracy  : 0.9155
F1 Score  : 0.9085
Precision : 0.9075
Recall    : 0.9155
Training time : 3483.21s
Model saved to /content/chicago_crimes/models/gbt_model


# MODEL 4: Linear SVM

In [ ]:
svm = LinearSVC(
    labelCol="arrest",
    featuresCol="features",
    maxIter=20,
    regParam=0.01
)

start     = time.time()
svm_model = svm.fit(train_df)
svm_preds = svm_model.transform(test_df)
svm_time  = round(time.time() - start, 2)

svm_results = evaluate_model(svm_preds, "Linear SVM")
svm_results["train_time_sec"] = svm_time
print(f"Training time : {svm_time}s")
save_model(svm_model, f"{MODELS_DIR}/svm_model")


Model     : Linear SVM
AUC-ROC   : 0.8512
Accuracy  : 0.9034
F1 Score  : 0.9009
Precision : 0.8991
Recall    : 0.9034
Training time : 54.46s
Model saved to /content/chicago_crimes/models/svm_model


In [ ]:
# Model Comparison Summary

print("MODEL COMPARISON SUMMARY")
results_df = pd.DataFrame([lr_results, rf_results, gbt_results, svm_results])
print(results_df.to_string(index=False))

# Save for Tableau Dashboard 2
results_df.to_csv(
    f"{TABLEAU_DIR}/tableau_model_comparison.csv", index=False
)
print("\nTableau Dashboard 2 file saved:")
print(f"  {TABLEAU_DIR}/tableau_model_comparison.csv")

MODEL COMPARISON SUMMARY
                 model    auc  accuracy     f1  precision  recall  train_time_sec
   Logistic Regression 0.8806    0.9181 0.9126     0.9112  0.9181           41.80
         Random Forest 0.8400    0.9232 0.9087     0.9255  0.9232          951.79
Gradient Boosted Trees 0.8616    0.9155 0.9085     0.9075  0.9155         3483.21
            Linear SVM 0.8512    0.9034 0.9009     0.8991  0.9034           54.46

Tableau Dashboard 2 file saved:
  /content/chicago_crimes/tableau/tableau_model_comparison.csv
